In [ ]:
!pip install crafter imageio imageio-ffmpeg --quiet
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import numpy as np
import math, os, copy, json, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch as T
import torch.optim as optim
from torch.distributions import Categorical
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import multiprocessing as mp
import crafter
import imageio

np.object = object
np.int = int
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

N_ACTIONS = 17  # crafter discrete action space
OBS_SHAPE  = (64, 64, 3)

print(f'Device: {DEVICE}')
print(f'Actions: {N_ACTIONS}')


## Parallel Crafter Env

In [ ]:
def _worker(seed, master_end, worker_end):
    master_end.close()
    env = crafter.Env(seed=seed)
    while True:
        cmd, data = worker_end.recv()
        if cmd == 'step':
            obs, reward, done, info = env.step(data)
            worker_end.send((obs, float(reward), bool(done), info))
        elif cmd == 'reset':
            obs = env.reset()
            worker_end.send(obs)
        elif cmd == 'close':
            worker_end.close(); break
        else:
            raise NotImplementedError

class ParallelCrafterEnv:
    def __init__(self, n_workers, base_seed=42):
        self.nenvs = n_workers
        self.closed = False
        self.workers = []
        master_ends, worker_ends = zip(*[mp.Pipe() for _ in range(n_workers)])
        self.master_ends = master_ends
        for i, (me, we) in enumerate(zip(master_ends, worker_ends)):
            p = mp.Process(target=_worker, args=(base_seed + i, me, we))
            p.daemon = True; p.start(); self.workers.append(p)
        for we in worker_ends: we.close()

    def reset(self):
        for me in self.master_ends: me.send(('reset', None))
        return np.stack([me.recv() for me in self.master_ends])  # (N,64,64,3)

    def step(self, actions):
        for me, a in zip(self.master_ends, actions): me.send(('step', int(a)))
        results = [me.recv() for me in self.master_ends]
        obs, rews, dones, infos = zip(*results)
        return np.stack(obs), np.array(rews, np.float32), np.array(dones, bool), list(infos)

    def reset_done(self, done_mask):
        """Reset only workers flagged True in done_mask. Returns full obs array;
        only done_mask=True slots contain fresh observations."""
        obs = np.zeros((self.nenvs, 64, 64, 3), np.uint8)
        for i, (me, d) in enumerate(zip(self.master_ends, done_mask)):
            if d:
                me.send(('reset', None))
        for i, (me, d) in enumerate(zip(self.master_ends, done_mask)):
            if d:
                obs[i] = me.recv()
        return obs

    def close(self):
        if self.closed: return
        for me in self.master_ends: me.send(('close', None))
        for w in self.workers: w.join()
        self.closed = True

print('ParallelCrafterEnv defined ✓')


## ResNet Encoder  (64×64×3 → 1024)

In [ ]:
class ResidualBlock(nn.Module):
    """Pre-activation residual block with layer norm."""
    def __init__(self, channels):
        super().__init__()
        self.ln = nn.LayerNorm([channels, 1, 1])  # spatial LayerNorm
        self.block = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
        )
    def forward(self, x):
        return x + self.block(x)


class CrafterResNet(nn.Module):
    """
    Large IMPALA-style ResNet matching Achievement Distillation paper.
    Channels: [64, 128, 128] — 3 stacks with 2 residual blocks each.
    64×64×3 → flatten 8192 → dense 256 → dense 1024 (latent rep).
    No LSTM, no transformer — pure CNN + MLP policy/value heads.

    Paper ref: Moon et al. NeurIPS 2023 (Achievement Distillation)
    """
    def __init__(self, n_actions, hidden_dim=1024):
        super().__init__()
        self.n_actions  = n_actions
        self.hidden_dim = hidden_dim

        def make_stack(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
                ResidualBlock(out_ch),
                ResidualBlock(out_ch),
            )

        # [64, 128, 128] channels — same as Achievement Distillation
        self.stack1 = make_stack(3,   64)   # 64→32
        self.stack2 = make_stack(64,  128)  # 32→16
        self.stack3 = make_stack(128, 128)  # 16→8
        # 128 × 8 × 8 = 8192

        self.fc = nn.Sequential(
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(8192, 256),
            nn.ReLU(),
            nn.Linear(256, hidden_dim),
            nn.ReLU(),
        )

        self.actor  = nn.Linear(hidden_dim, n_actions)
        self.critic = nn.Linear(hidden_dim, 1)

        # Orthogonal init for heads (as per paper)
        nn.init.orthogonal_(self.actor.weight,  gain=0.01)
        nn.init.orthogonal_(self.critic.weight, gain=0.1)
        nn.init.zeros_(self.actor.bias)
        nn.init.zeros_(self.critic.bias)

    def _encode(self, obs):
        if obs.dtype != torch.float32:
            obs = obs.float()
        obs = obs / 255.0
        obs = obs.permute(0, 3, 1, 2)
        return self.fc(self.stack3(self.stack2(self.stack1(obs))))

    def forward(self, inputs):
        obs, = inputs
        feat = self._encode(obs)

        act_logits = self.actor(feat)
        sm  = F.softmax(act_logits, dim=-1)
        lsm = F.log_softmax(act_logits, dim=-1)
        action   = Categorical(sm).sample().detach()
        log_prob = lsm.gather(1, action.unsqueeze(1))
        entropy  = -(lsm * sm).sum(1, keepdim=True)

        return self.critic(feat), action, log_prob, entropy

    def evaluate_actions(self, obs, actions):
        feat = self._encode(obs)
        lsm  = F.log_softmax(self.actor(feat), dim=-1)
        sm   = lsm.exp()
        return self.critic(feat), lsm.gather(1, actions.unsqueeze(1)), -(lsm*sm).sum(1, keepdim=True)

print("CrafterResNet defined ✓  ([64,128,128] IMPALA → 1024-dim latent)")


## Utils

In [ ]:
def compute_gae(boot_val, r_lst, mask_lst, values_lst, gamma, gae_lambda):
    """
    Generalized Advantage Estimation.
    boot_val  : (N,)  bootstrapped value at end of chunk
    r_lst     : list of (N,) rewards, oldest first
    mask_lst  : list of (N,) continuation masks (1=alive, 0=done)
    values_lst: list of (N,) value estimates, oldest first (detached)
    Returns   : advantages (T, N), returns (T, N)  — both on CPU
    """
    T = len(r_lst)
    N = boot_val.shape[0]
    advantages = np.zeros((T, N), np.float32)
    gae = np.zeros(N, np.float32)

    # bootstrap value for the step after the last one
    next_val = boot_val  # (N,)

    for t in reversed(range(T)):
        r    = r_lst[t]                          # (N,)
        mask = mask_lst[t]                       # (N,)
        v_t  = values_lst[t]                     # (N,)
        delta = r + gamma * next_val * mask - v_t
        gae   = delta + gamma * gae_lambda * mask * gae
        advantages[t] = gae
        next_val = v_t

    returns = advantages + np.stack(values_lst)  # (T, N)
    return advantages, returns                    # both (T, N) numpy


def render_video(model, episode_num, video_dir, device):
    """Record one eval episode. No LSTM state, no memory — just obs in, action out."""
    try:
        os.makedirs(video_dir, exist_ok=True)
        env    = crafter.Env(seed=9999)
        obs    = env.reset()
        frames = [obs.copy()]

        model.eval()
        done = False
        with torch.no_grad():
            while not done:
                obs_t = torch.from_numpy(obs[None].astype(np.float32)).to(device)
                _, action, _, _ = model((obs_t,))
                a = int(action.item())
                obs, reward, done, info = env.step(a)
                frames.append(obs.copy())

        out = os.path.join(video_dir, f'eval_ep{episode_num:05d}.mp4')
        imageio.mimwrite(out, [f.astype(np.uint8) for f in frames], fps=15)
        print(f'  Video saved: {out} ({len(frames)} frames)')
        model.train()
    except Exception as e:
        print(f'  Video skipped: {e}')
        model.train()
print('Utils defined OK')


## Episode Runner (epiplexity duel, bug-fixed)

In [ ]:
from collections import defaultdict

def run_one_episode(model, optimizer, envs, config, entropy_coeff, device, ewma_mean=0.0, ewma_std=1.0):
    """
    PPO episode runner for CrafterResNet (no LSTM).
    Collect nstep rollout, then ppo_nepoch x ppo_nbatch minibatch PPO updates.
    EWMA normalization applied to advantage (not returns) — value head predicts raw returns.
    Epiplexity = value_loss_before - value_loss_after on reference minibatch.
    """
    gamma      = config['gamma']
    val_coeff  = config['value-loss-weight']
    grad_clip  = config['grad-clip-norm']
    gae_lambda = config.get('gae-lambda', 0.65)
    n_workers  = config['n-workers']
    clip_param = config.get('clip-param', 0.2)
    ppo_nepoch = config.get('ppo-nepoch', 3)
    ppo_nbatch = config.get('ppo-nbatch', 8)
    nstep      = config['n-step-update']

    model.train()
    obs   = envs.reset()
    alive = np.ones(n_workers, bool)
    done  = np.zeros(n_workers, bool)

    obs_lst, act_lst, rew_lst, mask_lst = [], [], [], []
    logp_lst, val_lst, alive_lst        = [], [], []
    achievements_ever = defaultdict(lambda: np.zeros(n_workers, bool))
    episode_rewards   = np.zeros(n_workers)
    worker_steps      = np.zeros(n_workers, dtype=int)

    def _t(x): return torch.from_numpy(x).to(device)

    # ── Collect rollout ──────────────────────────────────────────────
    for _ in range(nstep):
        obs_t = _t(obs.astype(np.float32))
        with torch.no_grad():
            value_est, action, log_prob, _ = model((obs_t,))

        actions_np = action.cpu().numpy()
        next_obs, rewards, done, infos = envs.step(actions_np)

        alive_lst.append(alive.copy().astype(np.float32))
        episode_rewards += rewards * alive
        worker_steps    += (~done & alive).astype(int)

        for w, info in enumerate(infos):
            if not alive[w]: continue
            for ach, unlocked in info.get('achievements', {}).items():
                if unlocked:
                    achievements_ever[ach][w] = True

        obs_lst.append(obs.copy())
        act_lst.append(actions_np.copy())
        rew_lst.append(rewards.copy())
        mask_lst.append((~done).astype(np.float32))
        logp_lst.append(log_prob.detach().cpu().numpy().reshape(-1))
        val_lst.append(value_est.detach().cpu().numpy().reshape(-1))

        alive = alive & ~done
        obs   = next_obs
        if all(done): break

    # ── Bootstrap + GAE ─────────────────────────────────────────────
    with torch.no_grad():
        boot_val, _, _, _ = model((_t(obs.astype(np.float32)),))
    boot_np = boot_val.cpu().numpy().reshape(-1)
    adv_np, ret_np = compute_gae(boot_np, rew_lst, mask_lst, val_lst, gamma, gae_lambda)

    T = len(obs_lst)
    alive_w  = torch.tensor(np.stack(alive_lst).reshape(-1), dtype=torch.bool).to(device)
    obs_all  = torch.tensor(np.stack(obs_lst).reshape(T*n_workers,64,64,3), dtype=torch.float32).to(device)
    act_all  = torch.tensor(np.stack(act_lst).reshape(-1),                  dtype=torch.long).to(device)
    ret_all  = torch.tensor(ret_np.reshape(-1),                              dtype=torch.float32).to(device)
    adv_all  = torch.tensor(adv_np.reshape(-1),                              dtype=torch.float32).to(device)
    logp_old = torch.tensor(np.stack(logp_lst).reshape(-1),                  dtype=torch.float32).to(device)

    n_alive = alive_w.sum().item()
    if n_alive == 0:
        ach_success = {k: float(v.any()) for k, v in achievements_ever.items()}
        return {'reward_mean': float(episode_rewards.mean()), 'reward_sum': float(episode_rewards.sum()),
                'episode_epi': 0.0, 'n_updates': 0, 'crafter_score': 0.0, 'achievements': ach_success,
                'worker_steps': worker_steps.tolist(), 'steps_mean': float(worker_steps.mean()),
                'steps_min': int(worker_steps.min()), 'steps_max': int(worker_steps.max()),
                'env_steps': int(worker_steps.sum())}

    # ── EWMA advantage normalization ─────────────────────────────────
    # Normalize advantage using EWMA scale — value head predicts raw returns unchanged
    adv_alive = adv_all[alive_w]
    adv_normalized = (adv_all - ewma_mean) / ewma_std  # scale advantage by EWMA reward scale
    # Also normalize within batch for stability
    adv_batch = adv_normalized[alive_w]
    if adv_batch.numel() > 1:
        adv_normalized[alive_w] = (adv_batch - adv_batch.mean()) / (adv_batch.std(unbiased=False) + 1e-8)

    # ── Epiplexity reference batch ────────────────────────────────────
    alive_idx = torch.where(alive_w)[0]
    ref_size  = max(1, len(alive_idx) // ppo_nbatch)
    ref_idx   = alive_idx[:ref_size]

    with torch.no_grad():
        v_before, _, _ = model.evaluate_actions(obs_all[ref_idx], act_all[ref_idx])
        vl_before = F.smooth_l1_loss(v_before.reshape(-1), ret_all[ref_idx]).item()

    # ── PPO epochs ───────────────────────────────────────────────────
    batch_size = max(1, len(alive_idx) // ppo_nbatch)
    for epoch in range(ppo_nepoch):
        perm = alive_idx[torch.randperm(len(alive_idx))]
        for start in range(0, len(perm), batch_size):
            mb = perm[start:start+batch_size]
            if len(mb) == 0: continue

            val_new, logp_new, ent_new = model.evaluate_actions(obs_all[mb], act_all[mb])

            ratio  = torch.exp(logp_new.reshape(-1) - logp_old[mb])
            adv_mb = adv_normalized[mb]
            surr1  = ratio * adv_mb
            surr2  = torch.clamp(ratio, 1-clip_param, 1+clip_param) * adv_mb
            policy_loss = -torch.min(surr1, surr2).mean()
            value_loss  = F.smooth_l1_loss(val_new.reshape(-1), ret_all[mb]) * val_coeff
            ent_loss    = -entropy_coeff * ent_new.mean()
            loss        = policy_loss + value_loss + ent_loss

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

    # ── Epiplexity after ──────────────────────────────────────────────
    with torch.no_grad():
        v_after, _, _ = model.evaluate_actions(obs_all[ref_idx], act_all[ref_idx])
        vl_after = F.smooth_l1_loss(v_after.reshape(-1), ret_all[ref_idx]).item()

    episode_epi = vl_before - vl_after
    ach_success = {k: float(v.any()) for k, v in achievements_ever.items()}
    return {
        'reward_mean'  : float(episode_rewards.mean()),
        'reward_sum'   : float(episode_rewards.sum()),
        'episode_epi'  : float(episode_epi),
        'n_updates'    : ppo_nepoch * ppo_nbatch,
        'crafter_score': 0.0,
        'achievements' : ach_success,
        'worker_steps' : worker_steps.tolist(),
        'steps_mean'   : float(worker_steps.mean()),
        'steps_min'    : int(worker_steps.min()),
        'steps_max'    : int(worker_steps.max()),
        'env_steps'    : int(worker_steps.sum()),
    }

print("PPO runner defined OK")


## Main — Epiplexity Duel

In [ ]:
from collections import defaultdict

def main(config):
    device    = config['device']
    n_ep      = config['n-episodes']
    lr        = config['lr']
    ent_base  = config['entropy-weight']
    ent_low   = ent_base * 0.5
    ent_high  = ent_base * 2.0
    video_dir = config['video-dir']
    ckpt_dir  = config['ckpt-dir']
    os.makedirs(video_dir, exist_ok=True)
    os.makedirs(ckpt_dir,  exist_ok=True)

    writer = SummaryWriter(log_dir=config['log-dir'])

    agent_cfg = config['agent']
    base_model = CrafterResNet(N_ACTIONS, hidden_dim=agent_cfg['hidden-dim']).to(device)

    # EWMA tracks reward scale for advantage normalization
    ewma_mean  = 0.0
    ewma_var   = 1.0
    ewma_decay = config.get('ewma-decay', 0.99)
    total_env_steps = 0

    # ── Video test — catch renderer bugs before training starts ────────
    print('  🎬 Testing video renderer...')
    render_video(base_model, 0, video_dir, device)

    # ── Persistent base optimizer — state survives across episodes ─────
    base_opt = optim.AdamW(base_model.parameters(), lr=lr)

    wins_A = wins_B = 0
    best_score = 0.0
    start_ep = 0
    ALL_22_ACHIEVEMENTS_LIST = [
        'collect_coal','collect_diamond','collect_drink','collect_iron',
        'collect_sapling','collect_stone','collect_wood','defeat_skeleton',
        'defeat_zombie','eat_cow','eat_plant','make_iron_pickaxe',
        'make_iron_sword','make_stone_pickaxe','make_stone_sword',
        'make_wood_pickaxe','make_wood_sword','place_furnace','place_plant',
        'place_stone','place_table','wake_up',
    ]
    ach_counts     = {a: 0 for a in ALL_22_ACHIEVEMENTS_LIST}
    total_episodes = 0

    # ── Resume logic ───────────────────────────────────────────────────
    resume_mode = config.get('resume', None)  # None | 'weights_only' | 'full'
    if resume_mode:
        ckpt_dir_r = config['ckpt-dir']

        # find latest checkpoint automatically
        ckpts = sorted([
            f for f in os.listdir(ckpt_dir_r)
            if f.startswith('ckpt_ep') and f.endswith('.pt')
        ], key=lambda f: int(f.replace('ckpt_ep','').replace('.pt','')))

        if not ckpts:
            print('  ⚠️  No checkpoints found — starting fresh.')
        else:
            ckpt_path = os.path.join(ckpt_dir_r, ckpts[-1])
            ckpt = torch.load(ckpt_path, map_location=device)
            base_model.load_state_dict(ckpt['model'])
            print(f'  ✅ Loaded model weights from {ckpt_path}')

            if resume_mode == 'full' and 'optimizer' in ckpt:
                # full resume: restore optimizer state + counters
                base_opt.load_state_dict(ckpt['optimizer'])
                wins_A      = ckpt.get('wins_A', 0)
                wins_B      = ckpt.get('wins_B', 0)
                best_score  = ckpt.get('best_score', 0.0)
                start_ep    = ckpt.get('episode', 0) + 1
                print(f'  ✅ Restored optimizer state — resuming from ep {start_ep}')
            elif resume_mode == 'weights_only':
                # warm-start weights, fresh optimizer momentum
                print(f'  ✅ Weights-only resume — optimizer starts fresh (recommended after optimizer bug fix)')
            elif resume_mode == 'full' and 'optimizer' not in ckpt:
                print(f'  ⚠️  Checkpoint has no optimizer state — falling back to weights_only resume.')

    ALL_22_ACHIEVEMENTS = [
        'collect_coal','collect_diamond','collect_drink','collect_iron',
        'collect_sapling','collect_stone','collect_wood','defeat_skeleton',
        'defeat_zombie','eat_cow','eat_plant','make_iron_pickaxe',
        'make_iron_sword','make_stone_pickaxe','make_stone_sword',
        'make_wood_pickaxe','make_wood_sword','place_furnace','place_plant',
        'place_stone','place_table','wake_up'
    ]

    for epi in tqdm(range(start_ep, start_ep + n_ep)):

        # ── Fork model + optimizer for duel ───────────────────────────
        # Copy model weights
        model_A = copy.deepcopy(base_model)
        model_B = copy.deepcopy(base_model)

        # Copy optimizer state and rebind to each fork's parameters.
        # We must remap the param_groups so Adam's moment buffers
        # track the *new* model's parameters, not base_model's.
        def fork_optimizer(model_fork):
            opt = optim.AdamW(model_fork.parameters(), lr=lr)
            opt.load_state_dict(base_opt.state_dict())
            return opt

        opt_A = fork_optimizer(model_A)
        opt_B = fork_optimizer(model_B)

        env_A = ParallelCrafterEnv(agent_cfg['n-workers'], base_seed=42)
        env_B = ParallelCrafterEnv(agent_cfg['n-workers'], base_seed=42)

        stats_A = run_one_episode(model_A, opt_A, env_A, agent_cfg, ent_low,  device,
            ewma_mean=ewma_mean, ewma_std=float(np.sqrt(ewma_var+1e-8)))
        stats_B = run_one_episode(model_B, opt_B, env_B, agent_cfg, ent_high, device,
            ewma_mean=ewma_mean, ewma_std=float(np.sqrt(ewma_var+1e-8)))

        env_A.close(); env_B.close()

        # ── Epiplexity duel: winner updates base model + optimizer ─────
        epi_A = stats_A['episode_epi']
        epi_B = stats_B['episode_epi']
        # nan guard: nan comparison always returns False, corrupting duel
        if epi_A != epi_A: epi_A = 0.0
        if epi_B != epi_B: epi_B = 0.0

        if epi_A >= epi_B:
            base_model.load_state_dict(model_A.state_dict())
            base_opt.load_state_dict(opt_A.state_dict())
            wins_A += 1; winner = 'A (low-ent)'
            stats_w = stats_A
        else:
            base_model.load_state_dict(model_B.state_dict())
            base_opt.load_state_dict(opt_B.state_dict())
            wins_B += 1; winner = 'B (high-ent)'
            stats_w = stats_B

        total_env_steps += stats_w['env_steps']
        r = stats_w['reward_mean']
        ewma_mean = ewma_decay * ewma_mean + (1-ewma_decay) * r
        ewma_var  = ewma_decay * ewma_var  + (1-ewma_decay) * (r - ewma_mean)**2

        # ── Proper Crafter score — cumulative over ALL episodes (official metric) ──
        # Official metric: success rate = fraction of ALL training episodes
        # in which achievement was unlocked. NOT a rolling window.
        total_episodes += 1
        for a in ALL_22_ACHIEVEMENTS_LIST:
            if stats_w['achievements'].get(a, 0.0) > 0:
                ach_counts[a] += 1
        # Official score: exp(mean(log(1+si)))-1 over cumulative episode rates
        success_rates = {a: ach_counts[a]/total_episodes for a in ALL_22_ACHIEVEMENTS_LIST}
        si = np.array(list(success_rates.values())) * 100
        crafter_score = float(np.exp(np.mean(np.log(1 + si))) - 1)

        # ── TensorBoard ────────────────────────────────────────────────
        writer.add_scalar('epi/epiplexity_A',  stats_A['episode_epi'],   epi)
        writer.add_scalar('epi/epiplexity_B',  stats_B['episode_epi'],   epi)
        writer.add_scalar('perf/reward_winner',stats_w['reward_mean'],   epi)
        writer.add_scalar('perf/crafter_score_rolling100', crafter_score, epi)
        writer.add_scalar('meta/wins_A',       wins_A,                   epi)
        writer.add_scalar('meta/wins_B',       wins_B,                   epi)
        writer.add_scalar('steps/mean',        stats_w['steps_mean'],    epi)
        writer.add_scalar('steps/min',         stats_w['steps_min'],     epi)
        writer.add_scalar('steps/max',         stats_w['steps_max'],     epi)
        for ach, rate in success_rates.items():
            writer.add_scalar(f'achievements/{ach}', rate, epi)

        print(
            f'[Ep {epi:4d}] Winner={winner} | '
            f'EpiA={stats_A["episode_epi"]:+.4f} EpiB={stats_B["episode_epi"]:+.4f} | '
            f'Rew={stats_w["reward_mean"]:+.2f} | '
            f'Score={crafter_score:.4f} | '
            f'EnvSteps={total_env_steps/1e6:.3f}M | '
            f'Steps(min/avg/max)={stats_w["steps_min"]}/{stats_w["steps_mean"]:.0f}/{stats_w["steps_max"]} | '
            f'Wins={wins_A}/{wins_B}'
        )

        # ── Checkpoint + video every N episodes ────────────────────────
        if (epi + 1) % config['ckpt-every'] == 0:
            ckpt = os.path.join(ckpt_dir, f'ckpt_ep{epi+1}.pt')
            torch.save({
                'model'     : base_model.state_dict(),
                'optimizer' : base_opt.state_dict(),   # ← persisted optimizer
                'episode'   : epi,
                'best_score': best_score,
                'wins_A'    : wins_A,
                'wins_B'    : wins_B,
            }, ckpt)
            print(f'  💾 {ckpt}')

            if (epi + 1) % 200 == 0:
                render_video(base_model, epi+1, video_dir, device)

            try:
                import shutil
                drive_dir = f"/content/drive/MyDrive/crafter_epiplexity/{config['run-title']}"
                os.makedirs(drive_dir, exist_ok=True)
                shutil.copy2(ckpt, drive_dir)
                print(f'  ✅ Drive sync → {drive_dir}')
            except Exception as e:
                print(f'  ⚠️ Drive sync skipped: {e}')

        if stats_w['crafter_score'] > best_score:
            best_score = stats_w['crafter_score']
            torch.save({
                'model'    : base_model.state_dict(),
                'optimizer': base_opt.state_dict(),
            }, os.path.join(ckpt_dir, 'best_model.pt'))

    writer.close()
    print(f'\nDone. Best crafter score: {best_score:.4f}')
    return base_model

print('Main defined ✓')


## Config & Launch

In [ ]:
import os
os.makedirs('./runs/crafter_epiplexity_rand', exist_ok=True)
os.makedirs('./logs/crafter_epiplexity_rand', exist_ok=True)
os.makedirs('./videos/crafter_epiplexity_rand', exist_ok=True)

# V7: Big IMPALA ResNet — no LSTM, no transformer
HIDDEN_DIM  = 1024  # matches Achievement Distillation paper

# ── Resume mode ───────────────────────────────────────────────────────
# None         — start completely fresh
# 'weights_only' — load model weights from latest checkpoint, fresh optimizer
#                  USE THIS when resuming after the optimizer persistence fix
# 'full'         — load model + optimizer + counters (true resume, same run)
#                  USE THIS for normal Colab session restarts
RESUME_MODE = None   # ← change this as needed

config = {
    'run-title'    : 'crafter_epiplexity_v1',
    'device'       : DEVICE,
    'n-episodes'   : 5000,
    'lr'           : 3e-4,
    'entropy-weight': 0.01,
    'ewma-decay'   : 0.99,
    'ckpt-every'   : 100,
    'video-dir'    : './videos/crafter_epiplexity',
    'ckpt-dir'     : './runs/crafter_epiplexity',
    'log-dir'      : './logs/crafter_epiplexity',
    'resume'       : RESUME_MODE,  # None | 'weights_only' | 'full'

    'agent': {
        'n-workers'        : 4,
        'hidden-dim'       : HIDDEN_DIM,
        'n-actions'        : N_ACTIONS,
        'gamma'            : 0.95,
        'value-loss-weight': 0.5,
        'grad-clip-norm'   : 0.5,
        'n-step-update'    : 512,
        'gae-lambda'       : 0.65,
        'clip-param'       : 0.2,
        'ppo-nepoch'       : 3,
        'ppo-nbatch'       : 8,
    }
}

print('Config: V8 — PPO + IMPALA ResNet + EWMA adv norm + correct score')
print(f'  Workers: {config["agent"]["n-workers"]}')
print(f'  Device: {DEVICE}')

model = main(config)


In [ ]:
"""
Generate a high-resolution (256x256) eval video from a Crafter checkpoint.
Usage in Colab:
    exec(open('generate_eval_video.py').read())
    generate_video(ckpt_path='path/to/ckpt.pt', out_path='eval_hires.mp4', seed=42)
"""

import torch, crafter, imageio, numpy as np, os

def generate_video(ckpt_path, out_path='highres_video.mp4', seed=42, fps=15):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Load checkpoint — model is already defined in notebook scope
    ckpt = torch.load(ckpt_path, map_location=device)
    model = CrafterResNet(N_ACTIONS, hidden_dim=HIDDEN_DIM).to(device)
    model.load_state_dict(ckpt['model'])
    model.eval()

    # Two envs: agent sees 64x64, video renders at 256x256
    env_agent  = crafter.Env(seed=seed)
    env_render = crafter.Env(seed=seed, size=(256, 256))

    obs = env_agent.reset()
    env_render.reset()
    frames = [env_render.render()]
    done = False

    with torch.no_grad():
        while not done:
            obs_t = torch.from_numpy(obs[None].astype(np.float32)).to(device)
            _, action, _, _ = model((obs_t,))
            a = int(action.item())
            obs, _, done, _ = env_agent.step(a)
            env_render.step(a)
            frames.append(env_render.render())

    os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)
    imageio.mimsave(out_path, frames, fps=fps, quality=9)
    print(f"Saved {len(frames)} frames → {out_path}  ({os.path.getsize(out_path)//1024}KB)")
    return out_path

In [ ]:
# To generate higher res videos
# generate_video(ckpt_path='./runs/crafter_epiplexity/ckpt_ep1200.pt')